In [1]:
!pip install rapidfuzz --quiet

import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz, distance
from collections import Counter, defaultdict
import time


# Настройка отображения
pd.set_option('display.max_rows', 100)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.4 MB/s eta 0:00:0000:0100:01


In [2]:
print("=== ЗАГРУЗКА ЧИСТЫХ ДАННЫХ ===")
# Загружаем результаты прошлого ноутбука
orcs = pd.read_parquet('/kaggle/input/aim-orcs-precleaning/orcs_clean.parquet')
employees = pd.read_parquet('/kaggle/input/aim-orcs-precleaning/employees_clean.parquet')

print(f"Orcs: {orcs.shape}")
print(f"Employees: {employees.shape}")

=== ЗАГРУЗКА ЧИСТЫХ ДАННЫХ ===
Orcs: (47633, 7)
Employees: (2011759, 7)


In [ ]:
def build_vocab_stats(df_list, col):
    """
    Собирает все значения из колонки col во всех датафреймах
    и строит частотный словарь.
    """
    all_values = []
    for df in df_list:
        if col in df.columns:
            # Берем только непустые значения
            vals = df[col]
            vals = vals[vals != ""].tolist()
            all_values.extend(vals)

    # Считаем частоты
    counter = Counter(all_values)
    vocab_df = pd.DataFrame.from_dict(counter, orient='index', columns=['count']).reset_index()
    vocab_df = vocab_df.rename(columns={'index': 'word'})

    # Сортируем: сначала самые частые
    vocab_df = vocab_df.sort_values('count', ascending=False).reset_index(drop=True)

    return vocab_df

# Пример анализа для Имен
print("=== СТАТИСТИКА ПО ИМЕНАМ ===")
vocab_names = build_vocab_stats([orcs, employees], 'name')

print(f"Всего уникальных имен: {len(vocab_names)}")
print("Топ-10 частых:")
display(vocab_names.head(10))

print("Топ-10 редких (хвост):")
display(vocab_names.tail(10))

# Посмотрим на распределение (сколько имен встречаются 1 раз?)
singletons = vocab_names[vocab_names['count'] == 1].shape[0]
print(f"\nИмен, встречающихся ровно 1 раз: {singletons} ({singletons/len(vocab_names):.1%} от уникальных)")

=== СТАТИСТИКА ПО ИМЕНАМ ===
Всего уникальных имен: 218363
Топ-10 частых:


,word,count
0,александр,53133
1,сергей,45393
2,татьяна,43663
3,елена,41650
4,ольга,33978
5,наталья,33606
6,владимир,31756
7,алексей,29222
8,анна,28736
9,валентина,25853


Топ-10 редких (хвост):


,word,count
218353,дурние,1
218354,зинуиха,1
218355,армуллох,1
218356,а*еилна,1
218357,лилуя,1
218358,натал*ь,1
218359,гоухара,1
218360,хнатолий,1
218361,нуржигана,1
218362,мкьюя,1



Имен, встречающихся ровно 1 раз: 151173 (69.2% от уникальных)


In [ ]:
def generate_correction_map_waterfall(vocab_df, ### <--- 2. Имя функции упростили
                                      thresholds=[50000, 10000, 5000, 2000, 500, 100, 20, 0],
                                      min_ratio=9.0,
                                      cutoff_base=0.65,
                                      use_blocking=False, ### <--- 3. Флаг Блокировки
                                      use_delay=True):

    print(f"Запуск ВОДОПАДА (Cutoff {cutoff_base}, Ratio {min_ratio})...")

    correction_map = {}

    # 1. Инициализация
    top_threshold = thresholds[0]
    mask_clean = ~vocab_df['word'].str.contains(r'\*', regex=True)
    mask_len = vocab_df['word'].str.len() > 2

    current_targets_df = vocab_df[(vocab_df['count'] > top_threshold) & mask_clean & mask_len]
    current_targets = set(current_targets_df['word'].values)

    freq_dict = dict(zip(vocab_df['word'], vocab_df['count']))

    dormant_targets = []

    print(f"Базовых эталонов: {len(current_targets)}")

    # 2. Проход по уровням
    for i in range(len(thresholds) - 1):
        high = thresholds[i]
        low = thresholds[i+1]

        if use_delay:
            # Эталон должен быть больше жертвы в min_ratio раз. Макс. жертва сейчас = high.
            min_needed_freq = high * min_ratio

            # Переносим из dormant в active тех, кто проходит порог
            waking_up = [w for w in dormant_targets if freq_dict[w] >= min_needed_freq]
            dormant_targets = [w for w in dormant_targets if freq_dict[w] < min_needed_freq]

            current_targets.update(waking_up)
            if waking_up:
                print(f"  [Delay] Активировано {len(waking_up)} эталонов (Freq >= {min_needed_freq:.0f})")

        mask_range = (vocab_df['count'] > low) & (vocab_df['count'] <= high)
        candidates_df = vocab_df[mask_range & mask_len]
        all_candidates = candidates_df['word'].values

        if len(all_candidates) == 0: continue

        if use_blocking:
            targets_by_char = defaultdict(list)
            for t in current_targets:
                targets_by_char[t[0]].append(t)
            targets_list_all = list(current_targets) # Fallback для слов со звездочкой
        else:
            targets_list = list(current_targets)

        survivors = []
        fixed_count = 0

        for candidate in all_candidates:
            # ДИНАМИЧЕСКИЙ CUTOFF
            # Звезды всегда матчим чуть мягче (-0.05), так как * съедает информацию
            current_cutoff = (cutoff_base - 0.05) if '*' in candidate else cutoff_base

            if use_blocking:
                # Если в слове есть *, мы не знаем первую букву наверняка -> ищем везде
                if '*' in candidate:
                    targets_subset = targets_list_all
                else:
                    first_char = candidate[0]
                    targets_subset = targets_by_char.get(first_char, [])
            else:
                targets_subset = targets_list

            # Если список пуст, искать нечего
            if not targets_subset:
                if '*' not in candidate: survivors.append(candidate)
                continue

            match = process.extractOne(
                candidate,
                targets_subset,
                scorer=distance.DamerauLevenshtein.normalized_similarity,
                score_cutoff=current_cutoff
            )

            is_fixed = False

            if match:
                best_word, score, _ = match
                f_cand = freq_dict[candidate]
                f_targ = freq_dict[best_word]

                # Ratio Check
                ratio = f_targ / f_cand if f_cand > 0 else 9999

                if ratio >= min_ratio:
                    dist = distance.DamerauLevenshtein.distance(candidate, best_word)

                    # Distance Check
                    valid_dist = False
                    if '*' in candidate:
                        if dist == 1: valid_dist = True
                        elif dist == 2 and len(candidate) >= 5: valid_dist = True
                    else:
                        if dist == 1: valid_dist = True
                        elif dist == 2 and len(candidate) >= 6: valid_dist = True

                    if valid_dist:
                        correction_map[candidate] = {
                            'correct': best_word,
                            'dist': dist,
                            'score': score
                        }
                        is_fixed = True
                        fixed_count += 1

            if not is_fixed and '*' not in candidate:
                survivors.append(candidate)

        if use_delay:
            dormant_targets.extend(survivors)
        else:
            current_targets.update(survivors)
        print(f"  Уровень {low}-{high}: Исправлено {fixed_count}. Стали новыми эталонами: {len(survivors)}")

    print(f"Поиск завершен. Найдено {len(correction_map)} пар.")

    return correction_map

correction_map_names = generate_correction_map_waterfall(
    vocab_names,
    thresholds=[10000, 5000, 2000, 500, 100, 20, 0],
    min_ratio=9.0,
    cutoff_base=0.7,
    use_blocking=False,
    use_delay=True,
)

Запуск ВОДОПАДА (Cutoff 0.7, Ratio 9.0)...
Базовых эталонов: 35
  Уровень 5000-10000: Исправлено 0. Стали новыми эталонами: 23
  Уровень 2000-5000: Исправлено 18. Стали новыми эталонами: 30
  Уровень 500-2000: Исправлено 150. Стали новыми эталонами: 55
  [Delay] Активировано 28 эталонов (Freq >= 4500)
  Уровень 100-500: Исправлено 669. Стали новыми эталонами: 163
  [Delay] Активировано 43 эталонов (Freq >= 900)
  Уровень 20-100: Исправлено 889. Стали новыми эталонами: 779
  [Delay] Активировано 110 эталонов (Freq >= 180)
  Уровень 0-20: Исправлено 47220. Стали новыми эталонами: 145029
Поиск завершен. Найдено 48946 пар.


In [ ]:
def preview_correction_map_fast(vocab_df, correction_map, limit=100):
    if not correction_map:
        print("Карта пуста.")
        return

    print("Подготовка быстрого словаря частот...")
    # Превращаем DataFrame в словарь {слово: частота} -> это O(1) доступ
    freq_dict = dict(zip(vocab_df['word'], vocab_df['count']))

    print(f"Обработка {len(correction_map)} исправлений...")

    preview_data = []

    for wrong, data in correction_map.items():
        correct = data['correct']

        count_wrong = freq_dict.get(wrong, 0)
        count_correct = freq_dict.get(correct, 0)

        preview_data.append({
            'Original': wrong,
            'Fixed': correct,
            'Dist': data['dist'],
            'Freq_Orig': count_wrong,
            'Freq_Fix': count_correct,
            'Ratio': round(count_correct / count_wrong, 1) if count_wrong > 0 else 0
        })

    df_preview = pd.DataFrame(preview_data)

    # Сортируем: показываем те исправления, которые затронут больше всего строк (Freq_Orig)
    # И где дистанция минимальна
    df_preview = df_preview.sort_values(['Dist', 'Freq_Orig'], ascending=[True, False])

    print(f"=== ПРЕДПРОСМОТР ИСПРАВЛЕНИЙ (Топ-{limit}) ===")
    display(df_preview.head(limit))

    return df_preview

df_preview_names = preview_correction_map_fast(vocab_names, correction_map_names, limit=100)

Подготовка быстрого словаря частот...
Обработка 48946 исправлений...
=== ПРЕДПРОСМОТР ИСПРАВЛЕНИЙ (Топ-100) ===


,Original,Fixed,Dist,Freq_Orig,Freq_Fix,Ratio
0,олександр,александр,1,4274,53133,12.4
1,алена,елена,1,3458,41650,12.0
2,елено,елена,1,3441,41650,12.1
3,тотьяна,татьяна,1,3348,43663,13.0
4,ноталья,наталья,1,2761,33606,12.2
5,ольго,ольга,1,2721,33978,12.5
6,влодимир,владимир,1,2697,31756,11.8
7,владемир,владимир,1,2632,31756,12.1
8,олексей,алексей,1,2392,29222,12.2
9,онна,анна,1,2247,28736,12.8


In [ ]:
def audit_correction_ratios(vocab_df, correction_map, low_threshold=2.0):
    print(f"Аудит {len(correction_map)} исправлений...")

    # Быстрый словарь частот
    freq_dict = dict(zip(vocab_df['word'], vocab_df['count']))

    audit_data = []

    for wrong, data in correction_map.items():
        correct = data['correct']
        f_wrong = freq_dict.get(wrong, 0)
        f_correct = freq_dict.get(correct, 0)

        ratio = f_correct / f_wrong if f_wrong > 0 else 9999

        audit_data.append({
            'wrong': wrong,
            'correct': correct,
            'f_wrong': f_wrong,
            'f_correct': f_correct,
            'ratio': round(ratio, 2)
        })

    df_audit = pd.DataFrame(audit_data)

    # Статистика
    print(f"Мин. Ratio: {df_audit['ratio'].min()}")
    print(f"Макс. Ratio: {df_audit['ratio'].max()}")
    print(f"Медиана Ratio: {df_audit['ratio'].median()}")

    # Опасная зона
    risky = df_audit[df_audit['ratio'] < low_threshold]
    print(f"\n⚠️ Исправлений с Ratio < {low_threshold}: {len(risky)} шт.")

    if len(risky) > 0:
        print("Примеры 'на грани' (низкий перевес):")
        display(risky.sort_values('ratio').head(20))
    else:
        print("✅ Все исправления имеют сильный перевес частоты!")

    return df_audit

# АУДИТ
# Проверяем, исчезли ли ratio < 9.0 и появились ли 'никлой'/'мехаел'
_ = audit_correction_ratios(vocab_names, correction_map_names, low_threshold=10.0)


Аудит 48946 исправлений...
Мин. Ratio: 9.2
Макс. Ratio: 53133.0
Медиана Ratio: 4567.0

⚠️ Исправлений с Ratio < 10.0: 5 шт.
Примеры 'на грани' (низкий перевес):


,wrong,correct,f_wrong,f_correct,ratio
1737,прсковья,просковья,20,184,9.20
1752,дильбар,ильдар,20,185,9.25
1836,арменак,армен,19,180,9.47
1773,лара,клара,20,190,9.50
1797,леля,неля,19,185,9.74


In [ ]:
def apply_corrections_verbose(df, col, mapping, df_name="DataFrame"):
    """
    Применяет исправления и выводит статистику:
    - Сколько строк затронуто.
    - Как изменилось количество уникальных значений.
    """
    # 1. Превращаем "богатую" карту (с dist/score) в простую {bad: good}
    simple_map = {k: v['correct'] for k, v in mapping.items()}

    if not simple_map:
        print(f"⚠️ {df_name}: Карта исправлений пуста. Изменений нет.")
        return df

    # 2. Метрики ДО
    unique_before = df[col].nunique()

    # 3. Вычисляем маску (кого будем менять)
    mask = df[col].isin(simple_map.keys())
    affected_count = mask.sum()

    # 4. Применяем
    if affected_count > 0:
        # Используем .map только для нужных строк (быстро)
        df.loc[mask, col] = df.loc[mask, col].map(simple_map)

    # 5. Метрики ПОСЛЕ
    unique_after = df[col].nunique()
    diff = unique_before - unique_after

    # 6. Отчет
    print(f"=== ПРИМЕНЕНИЕ: {df_name} [{col}] ===")
    print(f"  📝 Строк изменено:      {affected_count}")
    print(f"  📊 Уникальных ДО:       {unique_before}")
    print(f"  📊 Уникальных ПОСЛЕ:    {unique_after}")
    if diff > 0:
        print(f"  📉 Сжатие словаря:      -{diff} (столько дублей устранено)")
    else:
        print(f"  😐 Словарь не изменился (странно, если были замены)")
    print("-" * 30)

    return df

# ПРИМЕНЯЕМ К ИМЕНАМ (correction_map_names уже готов)
orcs = apply_corrections_verbose(orcs, 'name', correction_map_names, "ORCS")
employees = apply_corrections_verbose(employees, 'name', correction_map_names, "EMPLOYEES")

=== ПРИМЕНЕНИЕ: ORCS [name] ===
  📝 Строк изменено:      6947
  📊 Уникальных ДО:       32398
  📊 Уникальных ПОСЛЕ:    28356
  📉 Сжатие словаря:      -4042 (столько дублей устранено)
------------------------------
=== ПРИМЕНЕНИЕ: EMPLOYEES [name] ===
  📝 Строк изменено:      494441
  📊 Уникальных ДО:       206517
  📊 Уникальных ПОСЛЕ:    158307
  📉 Сжатие словаря:      -48210 (столько дублей устранено)
------------------------------


In [ ]:
print("=== СТАТИСТИКА ПО ОТЧЕСТВАМ ===")
vocab_fath = build_vocab_stats([orcs, employees], 'fathername')
print(f"Всего уникальных: {len(vocab_fath)}")

# ГЕНЕРАЦИЯ (Водопад)
# Отчества - шаблонные. Ставим cutoff_base=0.75 (помягче),
# так как длина слова обычно большая.
correction_map_fath = generate_correction_map_waterfall(
    vocab_fath,
    thresholds=[10000, 5000, 2000, 500, 100, 20, 0],
    min_ratio=9.0,
    cutoff_base=0.67,
    use_blocking=False,
    use_delay=True,
)

# ПРЕВЬЮ
print("\n=== ПРЕДПРОСМОТР ОТЧЕСТВ ===")
_ = preview_correction_map_fast(vocab_fath, correction_map_fath, limit=100)

=== СТАТИСТИКА ПО ОТЧЕСТВАМ ===
Всего уникальных: 436579
Запуск ВОДОПАДА (Cutoff 0.67, Ratio 9.0)...
Базовых эталонов: 21
  Уровень 5000-10000: Исправлено 0. Стали новыми эталонами: 25
  Уровень 2000-5000: Исправлено 0. Стали новыми эталонами: 52
  Уровень 500-2000: Исправлено 182. Стали новыми эталонами: 95
  [Delay] Активировано 31 эталонов (Freq >= 4500)
  Уровень 100-500: Исправлено 805. Стали новыми эталонами: 230
  [Delay] Активировано 99 эталонов (Freq >= 900)
  Уровень 20-100: Исправлено 2132. Стали новыми эталонами: 565
  [Delay] Активировано 124 эталонов (Freq >= 180)
  Уровень 0-20: Исправлено 141442. Стали новыми эталонами: 238738
Поиск завершен. Найдено 144561 пар.

=== ПРЕДПРОСМОТР ОТЧЕСТВ ===
Подготовка быстрого словаря частот...
Обработка 144561 исправлений...
=== ПРЕДПРОСМОТР ИСПРАВЛЕНИЙ (Топ-100) ===


,Original,Fixed,Dist,Freq_Orig,Freq_Fix,Ratio
0,лександровна,александровна,1,1762,40751,23.1
1,сргеевна,сергеевна,1,1736,27555,15.9
2,ивновна,ивановна,1,1408,25733,18.3
3,сргеевич,сергеевич,1,1383,22693,16.4
4,иваовна,ивановна,1,1359,25733,18.9
5,иколаевна,николаевна,1,1288,29379,22.8
6,иановна,ивановна,1,1276,25733,20.2
7,николевна,николаевна,1,1230,29379,23.9
8,алексадровна,александровна,1,1188,40751,34.3
9,алксеевна,алексеевна,1,1153,20799,18.0


In [ ]:
def audit_risky_corrections(correction_map):
    print("=== АУДИТ РИСКОВАННЫХ ИСПРАВЛЕНИЙ ===")

    ogly_loss = []
    gender_swap = []

    for wrong, data in correction_map.items():
        correct = data['correct']

        # 1. Проверка потери Оглы/Кызы
        has_ogly_wrong = 'оглы' in wrong or 'кызы' in wrong
        has_ogly_right = 'оглы' in correct or 'кызы' in correct

        if has_ogly_wrong and not has_ogly_right:
            ogly_loss.append((wrong, correct))

        # 2. Проверка смены пола (Вич <-> Вна)
        # Грубая проверка по окончанию, но эффективная
        is_male_wrong = wrong.endswith('вич')
        is_female_wrong = wrong.endswith('вна')

        is_male_right = correct.endswith('вич')
        is_female_right = correct.endswith('вна')

        if (is_male_wrong and is_female_right) or (is_female_wrong and is_male_right):
            gender_swap.append((wrong, correct))

    print(f"⚠️ Потеря 'оглы/кызы': {len(ogly_loss)} шт.")
    if len(ogly_loss) > 0:
        print(f"   Примеры: {ogly_loss[:10]}")

    print(f"⚠️ Смена пола (вич/вна): {len(gender_swap)} шт.")
    if len(gender_swap) > 0:
        print(f"   Примеры: {gender_swap[:10]}")

    if len(ogly_loss) == 0 and len(gender_swap) == 0:
        print("✅ Критических ошибок не найдено.")

# Запуск аудита
audit_risky_corrections(correction_map_fath)

=== АУДИТ РИСКОВАННЫХ ИСПРАВЛЕНИЙ ===
⚠️ Потеря 'оглы/кызы': 0 шт.
⚠️ Смена пола (вич/вна): 0 шт.
✅ Критических ошибок не найдено.


In [10]:
# ПРИМЕНЯЕМ К отчествам (correction_map_names уже готов)
orcs = apply_corrections_verbose(orcs, 'fathername', correction_map_fath, "ORCS")
employees = apply_corrections_verbose(employees, 'fathername', correction_map_fath, "EMPLOYEES")

=== ПРИМЕНЕНИЕ: ORCS [fathername] ===
  📝 Строк изменено:      8889
  📊 Уникальных ДО:       39325
  📊 Уникальных ПОСЛЕ:    32674
  📉 Сжатие словаря:      -6651 (столько дублей устранено)
------------------------------
=== ПРИМЕНЕНИЕ: EMPLOYEES [fathername] ===
  📝 Строк изменено:      715213
  📊 Уникальных ДО:       412877
  📊 Уникальных ПОСЛЕ:    270474
  📉 Сжатие словаря:      -142403 (столько дублей устранено)
------------------------------


In [ ]:
print("=== СТАТИСТИКА ПО ФАМИЛИЯМ ===")
vocab_sur = build_vocab_stats([orcs, employees], 'surname')
print(f"Всего уникальных: {len(vocab_sur)}")

# ГЕНЕРАЦИЯ (Строгий режим)
# Cutoff 0.85 - чтобы не спутать "Иванов" и "Иванков"
# Ratio 9.0 - меняем только явный мусор на явных лидеров
correction_map_sur = generate_correction_map_waterfall(
    vocab_sur,
    thresholds=[10000, 5000, 2000, 500, 100, 20, 0],
    min_ratio=9.0,
    cutoff_base=0.75,
    use_blocking=True,
    use_delay=True,
)

# ПРЕВЬЮ
print("\n=== ПРЕДПРОСМОТР ФАМИЛИЙ (Топ-100) ===")
_ = preview_correction_map_fast(vocab_sur, correction_map_sur, limit=100)

=== СТАТИСТИКА ПО ФАМИЛИЯМ ===
Всего уникальных: 911732
Запуск ВОДОПАДА (Cutoff 0.75, Ratio 9.0)...
Базовых эталонов: 0
  Уровень 2000-5000: Исправлено 0. Стали новыми эталонами: 4
  Уровень 500-2000: Исправлено 0. Стали новыми эталонами: 127
  Уровень 100-500: Исправлено 0. Стали новыми эталонами: 1159
  [Delay] Активировано 46 эталонов (Freq >= 900)
  Уровень 20-100: Исправлено 917. Стали новыми эталонами: 6050
  [Delay] Активировано 565 эталонов (Freq >= 180)
  Уровень 0-20: Исправлено 94672. Стали новыми эталонами: 665188
Поиск завершен. Найдено 95589 пар.

=== ПРЕДПРОСМОТР ФАМИЛИЙ (Топ-100) ===
Подготовка быстрого словаря частот...
Обработка 95589 исправлений...
=== ПРЕДПРОСМОТР ИСПРАВЛЕНИЙ (Топ-100) ===


,Original,Fixed,Dist,Freq_Orig,Freq_Fix,Ratio
0,влкова,волкова,1,99,1619,16.4
1,ив*нова,иванова,1,98,2624,26.8
2,ивнаова,иванова,1,97,2624,27.1
3,мрозова,морозова,1,97,1394,14.4
4,иваонва,иванова,1,95,2624,27.6
6,и*анов,иванов,1,94,2060,21.9
7,иван*в,иванов,1,92,2060,22.4
8,иавнов,иванов,1,92,2060,22.4
9,ивано*а,иванова,1,90,2624,29.2
10,иванов*,иванова,1,90,2624,29.2


In [12]:
orcs = apply_corrections_verbose(orcs, 'surname', correction_map_sur, "ORCS")
employees = apply_corrections_verbose(employees, 'surname', correction_map_sur, "EMPLOYEES")

=== ПРИМЕНЕНИЕ: ORCS [surname] ===
  📝 Строк изменено:      3339
  📊 Уникальных ДО:       45391
  📊 Уникальных ПОСЛЕ:    42240
  📉 Сжатие словаря:      -3151 (столько дублей устранено)
------------------------------
=== ПРИМЕНЕНИЕ: EMPLOYEES [surname] ===
  📝 Строк изменено:      255738
  📊 Уникальных ДО:       885674
  📊 Уникальных ПОСЛЕ:    791158
  📉 Сжатие словаря:      -94516 (столько дублей устранено)
------------------------------


In [13]:
# 2. ФИНАЛЬНОЕ СОХРАНЕНИЕ
print("\n=== СОХРАНЕНИЕ CLEAN_V2 (FINAL) ===")

# Сохраняем в Parquet для следующего этапа (Deduplication)
orcs.to_parquet('orcs_clean_v2.parquet', index=False)
employees.to_parquet('employees_clean_v2.parquet', index=False)

print("Файлы v2 сохранены:")
print(f"1. orcs_clean_v2.parquet ({len(orcs)} строк)")
print(f"2. employees_clean_v2.parquet ({len(employees)} строк)")
print("\nЭтап 'Словарная нормализация' завершен.")
print("Теперь нажми 'Save Version' -> 'Save & Run All' и переходи к следующему ноутбуку.")


=== СОХРАНЕНИЕ CLEAN_V2 (FINAL) ===
Файлы v2 сохранены:
1. orcs_clean_v2.parquet (47633 строк)
2. employees_clean_v2.parquet (2011759 строк)

Этап 'Словарная нормализация' завершен.
Теперь нажми 'Save Version' -> 'Save & Run All' и переходи к следующему ноутбуку.
